In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt

def process_energy_data(filename, include_zero_mode=False):
    """
    Process energy data from especData.dat file.
    
    Parameters:
    -----------
    filename : str
        Path to the energy data file
    include_zero_mode : bool, optional
        Whether to include the zero mode (first mode) in calculations.
        Default is False (exclude zero mode).
        
    Returns:
    --------
    dict
        Dictionary containing processed energy data with keys:
        - 'times': array of time values
        - 'ke_data': 2D array of kinetic energy data
        - 'pe_data': 2D array of potential energy data  
        - 'dke_data': 2D array of kinetic energy dissipation data
        - 'ps_data': 2D array of pressure-strain data
        - 'total_ke': normalized total kinetic energy
        - 'total_pe': normalized total potential energy
        - 'total_energy': total energy (KE + PE)
        - 'total_dke': cumulative dissipation
        - 'total_ps': cumulative pressure-strain
        - 'time_steps': time step sizes
        - 'include_zero_mode': whether zero mode was included
    """
    # Load the data from the file
    with open(filename, 'r') as f:
        lines = f.readlines()

    # Process the data
    i = 0
    times = []
    ke_data = []
    pe_data = []
    dke_data = []
    ps_data = []
    
    while i < len(lines):
        current_set = {}
        
        while i < len(lines):
            line = lines[i].strip()
            if line:
                parts = line.split()
                label = parts[0].rstrip(':').upper()
                time_val = float(parts[1])
                data_vals = [float(x) for x in parts[2:]]
                
                current_set[label] = {'time': time_val, 'data': data_vals}
                i += 1
                
                # Check if we have a complete set (at minimum KE, PE, DKE, PS)
                if 'KE' in current_set and 'PE' in current_set and 'DKE' in current_set and 'PS' in current_set:
                    # Check if there's a PE2 in the next line
                    if i < len(lines):
                        next_line = lines[i].strip()
                        if next_line and next_line.split()[0].rstrip(':').upper() == 'PE2':
                            continue  # Read PE2 as well
                    break
            else:
                i += 1
        
        # Store the data from current set
        if 'KE' in current_set and 'PE' in current_set and 'DKE' in current_set and 'PS' in current_set:
            times.append(current_set['KE']['time'])
            ke_data.append(current_set['KE']['data'])
            pe_data.append(current_set['PE']['data'])
            dke_data.append(current_set['DKE']['data'])
            ps_data.append(current_set['PS']['data'])

    # Convert to numpy arrays
    ke_data = np.array(ke_data)
    pe_data = np.array(pe_data)
    dke_data = np.array(dke_data)
    ps_data = np.array(ps_data)
    times = np.array(times)

    # Calculate total energy - include or exclude zero mode based on parameter
    if include_zero_mode:
        total_ke = np.sum(ke_data, axis=1)  # Include all modes
        total_pe = np.sum(pe_data, axis=1)  # Include all modes
        total_dke_per_step = np.sum(dke_data, axis=1)  # Include all modes
        total_ps_per_step = np.sum(ps_data, axis=1)  # Include all modes
    else:
        total_ke = np.sum(ke_data[:, 1:], axis=1)  # Exclude the first mode
        total_pe = np.sum(pe_data[:, 1:], axis=1)  # Exclude the first mode
        total_dke_per_step = np.sum(dke_data[:, 1:], axis=1)  # Exclude the first mode
        total_ps_per_step = np.sum(ps_data[:, 1:], axis=1)  # Exclude the first mode
    
    total_ke = total_ke - total_ke[0]  # Normalize kinetic energy
    total_pe = total_pe - total_pe[0]  # Normalize potential energy

    total_energy = total_ke + total_pe

    # Calculate cumulative dissipation and pressure-strain using trapezoidal rule
    time_steps = np.diff(times)
    if len(time_steps) > 0:
        total_dke = np.zeros_like(total_dke_per_step)
        total_ps = np.zeros_like(total_ps_per_step)
        for i in range(1, len(total_dke_per_step)):
            dt = times[i] - times[i-1]
            total_dke[i] = total_dke[i-1] + 0.5 * (total_dke_per_step[i-1] + total_dke_per_step[i]) * dt
            total_ps[i] = total_ps[i-1] + 0.5 * (total_ps_per_step[i-1] + total_ps_per_step[i]) * dt
    else:
        total_dke = np.zeros_like(total_dke_per_step)
        total_ps = np.zeros_like(total_ps_per_step)
    
    time_steps = np.diff(times)
    time_steps = np.append(time_steps[0] if len(time_steps) > 0 else 0, time_steps)

    mode_info = "including zero mode" if include_zero_mode else "excluding zero mode"
    print(f"Detected data types: KE, PE, DKE, PS ({mode_info})")
    print(f"Number of time steps: {len(times)}")
    
    return {
        'times': times,
        'ke_data': ke_data,
        'pe_data': pe_data,
        'dke_data': dke_data,
        'ps_data': ps_data,
        'total_ke': total_ke,
        'total_pe': total_pe,
        'total_energy': total_energy,
        'total_dke': total_dke,
        'total_ps': total_ps,
        'time_steps': time_steps,
        'include_zero_mode': include_zero_mode
    }

In [ ]:
def plot_power_spectrums(data, time_instant, time_tolerance=1e-6):
    """
    Plot power spectrums of KE, PE, DKE, and PS for a specific time instant.
    
    Parameters:
    -----------
    data : dict
        Processed energy data from process_energy_data function
    time_instant : float
        Time instant to plot spectrums for
    time_tolerance : float, optional
        Tolerance for finding the closest time instant (default: 1e-6)
    """
    # Find the closest time index
    time_diffs = np.abs(data['times'] - time_instant)
    time_idx = np.argmin(time_diffs)
    
    if time_diffs[time_idx] > time_tolerance:
        print(f"Warning: Closest time found is {data['times'][time_idx]:.6f}, "
              f"which differs from requested {time_instant:.6f} by {time_diffs[time_idx]:.2e}")
    
    actual_time = data['times'][time_idx]
    
    # Extract data for this time instant
    ke_spectrum = data['ke_data'][time_idx, :]
    pe_spectrum = data['pe_data'][time_idx, :]
    dke_spectrum = data['dke_data'][time_idx, :]
    ps_spectrum = data['ps_data'][time_idx, :]
    
    # Create mode numbers (assuming data is ordered by mode)
    modes = np.arange(len(ke_spectrum))
    
    # Set matplotlib to use grayscale
    plt.rcParams['axes.prop_cycle'] = plt.cycler('color', ['black', 'gray', 'dimgray', 'lightgray', 'darkgray'])
    
    # Create the plot
    if data['include_zero_mode']:
        fig, axes = plt.subplots(1, 4, figsize=(24, 8))
    else:
        fig, axes = plt.subplots(1, 4, figsize=(24, 8))
    fig.suptitle(f'Power Spectrums at Time t = {actual_time:.6f}', fontsize=16, fontweight='bold', color='black')
    
    # KE Spectrum
    axes[0].semilogy(modes, np.abs(ke_spectrum), 'o-', color='black', linewidth=2, markersize=4)
    axes[0].set_xlabel('Mode Number', color='black')
    axes[0].set_ylabel('|KE|', color='black')
    axes[0].set_title('Kinetic Energy Spectrum', color='black')
    axes[0].grid(True, alpha=0.3, color='gray')
    axes[0].tick_params(colors='black')
    
    # PE Spectrum
    axes[1].semilogy(modes, np.abs(pe_spectrum), 's-', color='gray', linewidth=2, markersize=4)
    axes[1].set_xlabel('Mode Number', color='black')
    axes[1].set_ylabel('|PE|', color='black')
    axes[1].set_title('Potential Energy Spectrum', color='black')
    axes[1].grid(True, alpha=0.3, color='gray')
    axes[1].tick_params(colors='black')
    
    # DKE Spectrum
    axes[2].semilogy(modes, np.abs(dke_spectrum), '^-', color='dimgray', linewidth=2, markersize=4)
    axes[2].set_xlabel('Mode Number', color='black')
    axes[2].set_ylabel('|DKE|', color='black')
    axes[2].set_title('Kinetic Energy Dissipation Spectrum', color='black')
    axes[2].grid(True, alpha=0.3, color='gray')
    axes[2].tick_params(colors='black')
    
    # PS Spectrum
    axes[3].semilogy(modes, np.abs(ps_spectrum), 'd-', color='darkgray', linewidth=2, markersize=4)
    axes[3].set_xlabel('Mode Number', color='black')
    axes[3].set_ylabel('|PS|', color='black')
    axes[3].set_title('Pressure-Strain Spectrum', color='black')
    axes[3].grid(True, alpha=0.3, color='gray')
    axes[3].tick_params(colors='black')
    
    # Style all axes
    for ax in axes:
        for spine in ax.spines.values():
            spine.set_color('black')
            spine.set_linewidth(1.2)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\nSpectrum Statistics at t = {actual_time:.6f}:")
    print(f"KE total: {np.sum(ke_spectrum):.6e}, max: {np.max(np.abs(ke_spectrum)):.6e}")
    print(f"PE total: {np.sum(pe_spectrum):.6e}, max: {np.max(np.abs(pe_spectrum)):.6e}")
    print(f"DKE total: {np.sum(dke_spectrum):.6e}, max: {np.max(np.abs(dke_spectrum)):.6e}")
    print(f"PS total: {np.sum(ps_spectrum):.6e}, max: {np.max(np.abs(ps_spectrum)):.6e}")

In [ ]:
def plot_energy_analysis(data, time_percentage=1.0, print_table=False):
    """
    Plot comprehensive energy analysis including evolution over time, energy balance checks, and summary table.
    
    Energy balance equations:
    - dKE/dt = -(DKE + PS)
    - dPE/dt = +DKE
    - d(KE+PE)/dt = -PS
    
    Parameters:
    -----------
    data : dict
        Processed energy data from process_energy_data function
    time_percentage : float, optional
        Percentage of total time range to plot (default: 1.0 for 100%)
    print_table : bool, optional
        Whether to print detailed energy evolution table (default: False)
    """
    # Set matplotlib to use grayscale
    plt.rcParams['axes.prop_cycle'] = plt.cycler('color', ['black', 'gray', 'dimgray', 'lightgray', 'darkgray'])
    
    # Extract data from the dictionary
    times = data['times']
    total_ke = data['total_ke']
    total_pe = data['total_pe']
    total_energy = data['total_energy']
    total_dke = data['total_dke']
    total_ps = data['total_ps']
    total_include_zero_mode = data['include_zero_mode']
    
    # Calculate time range for plotting
    time_range_custom = times[0] + time_percentage * (times[-1] - times[0])
    time_mask_custom = times <= time_range_custom
    
    print(f"Showing data up to time: {time_range_custom:.3f} ({time_percentage*100:.1f}% of total range)")

    # Define consistent line styles for each type of data
    line_styles = {
        'actual': {'linestyle': '-', 'linewidth': 2.5, 'color': 'black'},           # Solid black - main data
        'reference': {'linestyle': '--', 'linewidth': 2.0, 'color': 'lightgray'},     # Dashed dark gray - reference/comparison
        'difference': {'linestyle': ':', 'linewidth': 2.5, 'color': 'black'},        # Dotted black - differences/residuals
        'negative': {'linestyle': '-', 'linewidth': 2.0, 'color': 'dimgray'}        # Solid dark gray - negative values
    }

    if total_include_zero_mode:
        fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    else:
        fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    
    # Adjust spacing to prevent overlapping
    padding = 0.08
    fig.subplots_adjust(top=1-padding, bottom=padding, left=padding, right=1-padding, hspace=0.2, wspace=0.15)
    fig.suptitle(f'Energy Balance Tracking', fontsize=20, fontweight='bold', color='black', y=0.98)

    # 1. Energy Evolution Over Time
    ax1 = axes[0, 0]
    ax1.plot(times, total_ke, label='ΔKE', **line_styles['actual'])
    ax1.plot(times, total_pe, label='ΔAPE', **line_styles['negative'])
    ax1.plot(times, -total_ps, label='∫PS dt', **line_styles['reference'])
    ax1.plot(times, total_energy+total_ps, label='Residual', **line_styles['difference'])
    ax1.set_xlabel('Time', color='black', fontsize=14)
    ax1.set_ylabel('Energy', color='black', fontsize=14)
    ax1.set_title('Energy Evolution', color='black', fontsize=15)
    # Place legend outside the plot area
    ax1.legend(frameon=True, fancybox=False, shadow=False, fontsize=11, 
               loc='upper left', bbox_to_anchor=(0.0, 1.0), ncol=2)
    ax1.grid(True, alpha=0.3, color='gray')
    ax1.tick_params(colors='black', labelsize=12)

    # Auto-adjust y-limits for subplot 1
    y_data_ax1 = np.concatenate([total_ke[time_mask_custom], total_pe[time_mask_custom], (total_energy+total_ps)[time_mask_custom]])
    y_data_ax1 = y_data_ax1[np.isfinite(y_data_ax1)]  # Remove any inf/nan values
    if len(y_data_ax1) > 0:
        y_min, y_max = np.min(y_data_ax1), np.max(y_data_ax1)
        y_range = y_max - y_min
        margin = 0.1 * y_range if y_range > 0 else 0.1 * abs(y_max)
        ax1.set_ylim(y_min - margin, y_max + margin)

    # 2. KE Balance: dKE/dt = -(DKE + PS)
    ax2 = axes[0, 1]
    ax2.plot(times, total_ke - total_ke[0], label='ΔKE', **line_styles['actual'])
    ax2.plot(times, total_dke - total_ps, label='∫(PS - DKE)dt', **line_styles['reference'])
    difference_ke = (total_ke - total_ke[0]) - (total_dke - total_ps)
    ax2.plot(times, difference_ke, label='Residual', **line_styles['difference'])
    ax2.set_xlabel('Time', color='black', fontsize=14)
    ax2.set_ylabel('Energy', color='black', fontsize=14)
    ax2.set_title('KE Balance', color='black', fontsize=15)
    ax2.legend(frameon=True, fancybox=False, shadow=False, fontsize=11,
               loc='upper left', bbox_to_anchor=(0.0, 1.0), ncol=3)
    ax2.grid(True, alpha=0.3, color='gray')
    ax2.tick_params(colors='black', labelsize=12)

    # Auto-adjust y-limits for subplot 2
    y_data_ax2 = np.concatenate([(total_ke-total_ke[0])[time_mask_custom], (total_dke - total_ps)[time_mask_custom], difference_ke[time_mask_custom]])
    y_data_ax2 = y_data_ax2[np.isfinite(y_data_ax2)]
    if len(y_data_ax2) > 0:
        y_min, y_max = np.min(y_data_ax2), np.max(y_data_ax2)
        y_range = y_max - y_min
        margin = 0.1 * y_range if y_range > 0 else 0.1 * abs(y_max)
        ax2.set_ylim(y_min - margin, y_max + margin)

    # 3. PE Balance: daPE/dt = +DKE
    ax3 = axes[1, 0]
    ax3.plot(times, total_pe - total_pe[0], label='ΔAPE', **line_styles['actual'])
    ax3.plot(times, -total_dke, label='∫DKE dt', **line_styles['reference'])
    difference_pe = (total_pe - total_pe[0]) - (-total_dke)
    ax3.plot(times, difference_pe, label='Residual', **line_styles['difference'])
    ax3.set_xlabel('Time', color='black', fontsize=14)
    ax3.set_ylabel('Energy', color='black', fontsize=14)
    ax3.set_title('APE Balance', color='black', fontsize=15)
    ax3.legend(frameon=True, fancybox=False, shadow=False, fontsize=11,
               loc='upper left', bbox_to_anchor=(0.0, 1.0), ncol=3)
    ax3.grid(True, alpha=0.3, color='gray')
    ax3.tick_params(colors='black', labelsize=12)

    # Auto-adjust y-limits for subplot 3
    y_data_ax3 = np.concatenate([(total_pe - total_pe[0])[time_mask_custom], (-total_dke)[time_mask_custom], difference_pe[time_mask_custom]])
    y_data_ax3 = y_data_ax3[np.isfinite(y_data_ax3)]
    if len(y_data_ax3) > 0:
        y_min, y_max = np.min(y_data_ax3), np.max(y_data_ax3)
        y_range = y_max - y_min
        margin = 0.1 * y_range if y_range > 0 else 0.1 * abs(y_max)
        ax3.set_ylim(y_min - margin, y_max + margin)

    # 4. Total Energy Balance: d(KE+APE)/dt = -PS
    ax4 = axes[1, 1]
    ax4.plot(times, total_energy - total_energy[0], label='Δ(KE+APE)', **line_styles['actual'])
    ax4.plot(times, -total_ps, label='∫PS dt', **line_styles['reference'])
    difference_total = (total_energy - total_energy[0]) - (-total_ps)
    ax4.plot(times, difference_total, label='Residual', **line_styles['difference'])
    ax4.set_xlabel('Time', color='black', fontsize=14)
    ax4.set_ylabel('Energy', color='black', fontsize=14)
    ax4.set_title('Total Energy Balance', color='black', fontsize=15)
    ax4.legend(frameon=True, fancybox=False, shadow=False, fontsize=11,
               loc='upper left', bbox_to_anchor=(0.0, 1.0), ncol=3)
    ax4.grid(True, alpha=0.3, color='gray')
    ax4.tick_params(colors='black', labelsize=12)

    # Auto-adjust y-limits for subplot 4
    y_data_ax4 = np.concatenate([(total_energy - total_energy[0])[time_mask_custom], (-total_ps)[time_mask_custom], difference_total[time_mask_custom]])
    y_data_ax4 = y_data_ax4[np.isfinite(y_data_ax4)]
    if len(y_data_ax4) > 0:
        y_min, y_max = np.min(y_data_ax4), np.max(y_data_ax4)
        y_range = y_max - y_min
        margin = 0.1 * y_range if y_range > 0 else 0.1 * abs(y_max)
        ax4.set_ylim(y_min - margin, y_max + margin)

    # 5. zero-mode KE (only if excluding zero mode)
    if not total_include_zero_mode:
        ke_data = data['ke_data'][:,0]
        dke_data = data['dke_data'][:,0]
        ps_data = data['ps_data'][:,0]
        int_dke_ps = np.cumsum((dke_data + ps_data) * data['time_steps'])
        ax5 = axes[0, 2]
        ax5.plot(times, ke_data[time_mask_custom]-ke_data[0], label='Zero-mode ΔKE', **line_styles['actual'])
        ax5.plot(times, -int_dke_ps[time_mask_custom], label='-∫(DKE+PS)dt', **line_styles['reference'])
        ax5.set_xlabel('Time', color='black', fontsize=14)
        ax5.set_ylabel('Energy', color='black', fontsize=14)
        ax5.set_title('Zero-mode KE', color='black', fontsize=15)
        ax5.legend(frameon=True, fancybox=False, shadow=False, fontsize=11,
                   loc='upper right', bbox_to_anchor=(1.0, 1.0), ncol=1)
        ax5.grid(True, alpha=0.3, color='gray')
        ax5.tick_params(colors='black', labelsize=12)
        
        # Calculate decay rate as percentage drop per time step
        ke_fit = ke_data[time_mask_custom]        
        if len(ke_fit) > 1 and ke_fit[0] != 0:
            percentage_losses = []
            for i in range(1, len(ke_fit)):
                if ke_fit[i-1] != 0:  # Avoid division by zero
                    loss_percent = (ke_fit[i-1] - ke_fit[i]) / ke_fit[i-1] * 100
                    percentage_losses.append(loss_percent)
            
            # Take average of all percentage losses
            if percentage_losses:
                avg_decay_rate = np.mean(percentage_losses)
                textstr = f'Avg decay rate: {avg_decay_rate:.3f}%/step'
            else:
                textstr = 'Decay rate: N/A (no valid steps)'
        else:
            textstr = 'Decay rate: N/A'
            
        props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='black')
        ax5.text(0.05, 0.05, textstr, transform=ax5.transAxes, fontsize=11,
            verticalalignment='bottom', bbox=props, color='black')

        # 6. Cumulative terms comparison
        ax6 = axes[1, 2]
        ax6.plot(times, total_dke, label='∫DKE dt', **line_styles['actual'])
        ax6.plot(times, total_ps, label='∫PS dt', **line_styles['reference'])
        ax6.plot(times, total_dke + total_ps, label='∫(DKE+PS)dt', **line_styles['difference'])
        ax6.set_xlabel('Time', color='black', fontsize=14)
        ax6.set_ylabel('Energy', color='black', fontsize=14)
        ax6.set_title('Cumulative Terms', color='black', fontsize=15)
        ax6.legend(frameon=True, fancybox=False, shadow=False, fontsize=11,
                   loc='upper left', bbox_to_anchor=(0.0, 1.0), ncol=1)
        ax6.grid(True, alpha=0.3, color='gray')
        ax6.tick_params(colors='black', labelsize=12)

    # Set all axes to have black spines and specified time range
    for ax in axes.flat:
        for spine in ax.spines.values():
            spine.set_color('black')
            spine.set_linewidth(1.2)
        # Set x-axis to specified percentage of time range
        ax.set_xlim(times[0], time_range_custom)

    plt.show()

    if print_table:
        # Time stamp table with grayscale styling
        detailed_data = []
        for i, t in enumerate(times):
            row = {
                'Time': f'{t:.4f}',
                'KE': f'{total_ke[i]:.6e}',
                'PE': f'{total_pe[i]:.6e}',
                'Total Energy': f'{total_energy[i]:.6e}',
                'Cumul. DKE': f'{total_dke[i]:.6e}',
                'Cumul. PS': f'{total_ps[i]:.6e}',
                'KE Residual': f'{difference_ke[i]:.6e}',
                'PE Residual': f'{difference_pe[i]:.6e}',
                'Total Residual': f'{difference_total[i]:.6e}'
            }
            detailed_data.append(row)

        df = pd.DataFrame(detailed_data)

        styled_df = df.style.set_properties(**{
            'text-align': 'center',
            'font-size': '10px'
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#40466e'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
        ])

        print("\n" + "="*80)
        print("ENERGY ANALYSIS SUMMARY")
        print("="*80)
        print(f"Initial KE: {total_ke[0]:.6e}")
        print(f"Final KE: {total_ke[-1]:.6e}")
        print(f"KE Change: {total_ke[-1] - total_ke[0]:.6e}")
        print(f"Cumulative (DKE+PS): {-(total_dke[-1] + total_ps[-1]):.6e}")
        print(f"KE Balance Residual: {difference_ke[-1]:.6e}")
        print()
        print(f"Initial PE: {total_pe[0]:.6e}")
        print(f"Final PE: {total_pe[-1]:.6e}")
        print(f"PE Change: {total_pe[-1] - total_pe[0]:.6e}")
        print(f"Cumulative DKE: {total_dke[-1]:.6e}")
        print(f"PE Balance Residual: {difference_pe[-1]:.6e}")
        print()
        print(f"Total Energy Change: {total_energy[-1] - total_energy[0]:.6e}")
        print(f"Cumulative PS: {-total_ps[-1]:.6e}")
        print(f"Total Energy Balance Residual: {difference_total[-1]:.6e}")
        print("="*80)
        print("DETAILED ENERGY EVOLUTION TABLE")
        print("="*80)
        print(f"Total time steps: {len(times)}")
        display(styled_df)

In [ ]:
# Over K
print("Processing energy data K...")

energy_data_K = process_energy_data('../../output/especData_K.dat',include_zero_mode=True)
plot_energy_analysis(energy_data_K, time_percentage=1.0)
time_data_K = energy_data_K['times']
plot_power_spectrums(energy_data_K, time_instant = time_data_K[1])
plot_power_spectrums(energy_data_K, time_instant = time_data_K[-1])

In [ ]:
# over M
print("Processing energy data M...")

energy_data_M = process_energy_data('../../output/especData_M.dat',include_zero_mode=True)
plot_energy_analysis(energy_data_M, time_percentage=1.0)
time_data_M = energy_data_M['times']
plot_power_spectrums(energy_data_M, time_instant = time_data_M[1])
plot_power_spectrums(energy_data_M, time_instant = time_data_M[-1])